# Notebook 17 — what are the 300 scored items actually made of?

`transcription_correct` is an exact match between two canonicalized strings, so
the perception AUROC rests on that comparison being fair. This notebook opens
the comparison up, classifies **every** item by why it scored the way it did,
and shows the handwriting behind each category.

For every item the scoring pipeline runs four stages, on the model samples and
on the ground truth alike:

| stage | function | produces |
|---|---|---|
| 1 | *(the run)* | raw model output |
| 2 | `pilot.parsing.parse_transcription` | the `**Answer:**` field |
| 3 | `pilot.canonicalize.extract_final_answer` | the final-answer span |
| 4 | `pilot.rescore.answer_label` | the string actually compared |

Offline inspection found four things worth seeing rather than taking on
trust, three of them genuine bugs:

1. **A real bug.** `structural_clean` unwraps `\textcolor{}{}` with a `[^{}]*`
   body, so nested cases survive into the label. 85/300 ground truths, 59 of
   them `has_error=1` — FERMAT marks the *injected error* in red.
2. **A second real bug.** `extract_final_answer`'s last-line tier splits on
   `.` as a sentence terminator, so it also splits decimal numbers:
   `"the area is 75.46 cm."` extracts as `"46 cm"`.
3. **A third real bug.** `parse_latex` does not fail loudly on input it only
   partly understands — it parses a **prefix** and silently returns it.
   `"Hence, the required number of words is 24"` becomes `h*(e*(n*(c*e)))`
   (SymPy read "Hence" as five multiplied variables and threw the 24 away);
   `"40^\circ 20' = \frac{121\pi}{540}"` becomes `40**circ*20`, dropping the
   `=` and the entire answer. 37/300 ground truths, 44/1500 samples.
4. **Formatting and scope mismatches.** `2^3 = 8` vs `2^{3} = 8`; `0 = 9` vs
   `0 = 9,`; and the model reporting `= 75.46 cm^2` where the ground truth
   spells out `= \pi r^2 = ... = 75.46 cm^2`.

Bugs 2 and 3 are why they had to be *fixed* rather than noted. Bug 3 is the
worst because it **collapses**: `\frac{1210}{540}` and `\frac{121\pi}{540}`
are different answers that reduce to the same label, which deflates entropy as
well as manufacturing matches. Fixing them *lowers* accuracy — the honest
direction.

**No GPU, no model.** Reads two results CSVs and the FERMAT images. ~5 min.

**What this notebook does NOT do:** change the headline. `strict_v1` stays the
frozen rule of record — it produced every locked result and all of
`reference/*.json`. The looser rules are reported *alongside* it as a
sensitivity analysis. Moving a scoring rule after seeing that it raises
accuracy is the move this project keeps refusing to make.

In [1]:
# Auth + code access. No GPU/model needed -- this notebook only reads the
# dataset and existing results CSVs, it never runs generation.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"

# Reuses the token already cached on Drive by earlier notebooks.
with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
# antlr4 pin: without it SymPy's LaTeX parser fails at CALL time, silently
# degrading every label to the plain-text tier. See
# pilot.canonicalize.latex_parser_available -- this cost 43/300 items once.
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))

# Purge any pilot.* left over from a previous clone in this runtime.
# importlib.invalidate_caches() does NOT reload already-imported modules, and
# a stale one produced a KeyError on the 2026-08-08 notebook-13 run for a
# symbol that was demonstrably on disk.
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.canonicalize
import pilot.data
import pilot.parsing
import pilot.plotting
import pilot.rescore

print(f"pilot package imported from: {os.path.dirname(pilot.rescore.__file__)}")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working. Every label falls back to plain text, "
    "which inflates entropy and deflates accuracy, and the numbers below will "
    "not match the offline analysis. Fix the antlr4 pin before continuing."
)
print("SymPy LaTeX parser: OK")

Mounted at /content/drive
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 15.7 MB/s eta 0:00:00
  Building editable for pilot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
pilot package imported from: /content/repo/pilot
SymPy LaTeX parser: OK


## 1. Load both runs and rebuild the sample

Two models on the **same 300 images**: Qwen2.5-VL-3B (the run every scoring
number is quoted from) and Pixtral-12B (the confirmed second perception
family). Running the classification on both is what turns "the extractor has
problems" into "the extractor has problems that are not Qwen-specific".

The CSVs carry every raw model sample, so the whole scoring chain can be
re-run offline. The images are not in the CSV — they come from FERMAT, which
is gated, which is why this notebook runs in Colab rather than locally.

The `load_fermat_balanced` call reproduces the *same* draw both runs used, and
the assert below checks that row *i* of each CSV really is item *i* of the
sample rather than trusting the order.

In [2]:
import ast

import pandas as pd

RUNS = {
    "Qwen2.5-VL-3B": "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv",
    "Pixtral-12B":   "pixtral_perception_full_n300_pixtral-12b_20260809T211028Z.csv",
}
SEED, N_ITEMS, ERROR_FRAC = 42, 300, 0.5

runs = {}
for name, fname in RUNS.items():
    runs[name] = pd.read_csv(f"{RESULTS_DIR}/{fname}")
    print(f"{name:15s} {len(runs[name])} rows, "
          f"model={runs[name]['model_id'].unique().tolist()}, "
          f"K={runs[name]['k_transcription'].unique().tolist()}")

sample = pilot.data.load_fermat_balanced(
    n=N_ITEMS, seed=SEED, target_error_frac=ERROR_FRAC)

# load_fermat_balanced SHUFFLES its final selection, so index alignment is an
# assumption to verify, not one to make. Checking the question text pins the
# row-to-image mapping every display below depends on -- for BOTH runs, since
# the viewer takes images from the sample and text from whichever CSV.
for name, df in runs.items():
    assert len(sample) == len(df), f"{name}: {len(sample)} items vs {len(df)} rows"
    bad = [i for i in range(len(df))
           if sample[i]["orig_q"].strip() != str(df.iloc[i]["orig_q"]).strip()]
    assert not bad, (
        f"{name}: {len(bad)} rows where the rebuilt sample's question does not "
        f"match the CSV's (first: {bad[:5]}). Images would be attached to the "
        "wrong rows -- do not trust anything past this cell until resolved.")
print("\nsample order matches both CSVs on all 300 rows -- images are index-aligned")

Qwen2.5-VL-3B   300 rows, model=['Qwen/Qwen2.5-VL-3B-Instruct'], K=[5]
Pixtral-12B     300 rows, model=['mistral-community/pixtral-12b'], K=[5]


README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  481MB            

...

data/train-00008-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00009-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  493MB            

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2244 [00:00<?, ? examples/s]


sample order matches both CSVs on all 300 rows -- images are index-aligned


## 2. The sensitivity table

Four cumulative rules (see `pilot/rescore.py` for the full definitions):

- **`strict_v1`** — exactly `canonical_answer_label`. Frozen; the rule of record.
- **`fixed_v2`** — v1 with all three extractor bugs corrected. *Bug fixes*, so
  they move items **both ways** and can lower accuracy.
- **`relaxed_v3`** — v2 ignoring formatting the mathematics does not depend on.
- **`final_term_v4`** — v3 reduced to the term after the last top-level `=`.

v4 changes the **label**, not just the comparison, so entropy and correctness
stay derived from the same representation. Scoring correctness leniently while
leaving entropy strict would compare two different objects.

In [3]:
# ~4 min: re-runs the full parse -> extract -> canonicalize chain for
# 2 models x 300 items x 5 samples x 4 rules, with SymPy on every label.
sens = {}
for name, df in runs.items():
    print(f"=== {name} ===")
    s = pilot.rescore.scoring_sensitivity(df, n_boot=10000, seed=0, progress=True)
    sens[name] = s
    view = s.copy()
    view["accuracy"] = (view["accuracy"] * 100).round(1).astype(str) + "%"
    view["AUROC [95% CI]"] = [f"{r.auroc:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]"
                              for r in s.itertuples()]
    print(view[["rule", "n_correct", "accuracy", "AUROC [95% CI]",
                "excludes_chance", "n_at_max_entropy"]].to_string(index=False))
    print(f"  accuracy {s.accuracy.iloc[0]:.1%} -> {s.accuracy.iloc[-1]:.1%}   "
          f"AUROC {s.auroc.iloc[0]:.3f} -> {s.auroc.iloc[-1]:.3f}   "
          f"every rule excludes chance: {bool(s.excludes_chance.all())}\n")

=== Qwen2.5-VL-3B ===
[1/4] strict_v1: rescoring 300 items...


strict_v1:   0%|          | 0/300 [00:00<?, ?it/s]

          bootstrapping 10000 resamples...
[2/4] fixed_v2: rescoring 300 items...


fixed_v2:   0%|          | 0/300 [00:00<?, ?it/s]

          bootstrapping 10000 resamples...
[3/4] relaxed_v3: rescoring 300 items...


...

          bootstrapping 10000 resamples...
[3/4] relaxed_v3: rescoring 300 items...


relaxed_v3:   0%|          | 0/300 [00:00<?, ?it/s]

          bootstrapping 10000 resamples...
[4/4] final_term_v4: rescoring 300 items...


final_term_v4:   0%|          | 0/300 [00:00<?, ?it/s]

          bootstrapping 10000 resamples...
         rule  n_correct accuracy       AUROC [95% CI]  excludes_chance  n_at_max_entropy
    strict_v1        125    41.7% 0.828 [0.782, 0.871]             True                51
     fixed_v2        128    42.7% 0.851 [0.807, 0.890]             True                63
   relaxed_v3        140    46.7% 0.830 [0.785, 0.873]             True                52
final_term_v4        163    54.3% 0.807 [0.758, 0.854]             True                26
  accuracy 41.7% -> 54.3%   AUROC 0.828 -> 0.807   every rule excludes chance: True



**Read it this way.** Accuracy moves a lot — the strict rule really is
undercounting correct reads. The AUROC decays gracefully and never touches
chance. A signal that existed only because of a pedantic string comparison
would not do that, so the perception result belongs to the entropy rather than
to the comparison.

## 3. Classify every item

The four illustrative buckets this notebook used to draw examples from
**overlapped** — an item could be both a cosmetic mismatch and
extractor-tier-unstable — so they could show you a case of X but could not say
what the population is made of. These seven categories are mutually exclusive
and cover every item.

| category | meaning |
|---|---|
| `correct_robust` | correct under the frozen rule **and** after the bug fixes |
| `bug_fix_recovered` | wrong under the frozen rule; a bug fix recovered it |
| `cosmetic_mismatch` | spacing, punctuation, braces, currency |
| `scope_mismatch` | model gave the answer, truth gave the whole chain |
| `false_pass_removed` | **looked correct only because a bug mangled both sides** |
| `broken_by_relaxation` | correct earlier, lost when a looser rule changed the vote |
| `genuinely_wrong` | wrong under every rule |

The last three exist because **the rules are not monotone**. A naive
cumulative scheme would hide them, and `false_pass_removed` is the one to read
first — it is where relaxing a comparison manufactures a wrong answer.

In [4]:
classified, summaries = {}, []
for name, df in runs.items():
    c = pilot.rescore.classify_scoring_outcome(df, progress=True)
    classified[name] = c
    summaries.append(pilot.rescore.scoring_category_summary(c, label=name))
    assert len(c) == len(df) and c["category"].notna().all()

summary = pd.concat(summaries, ignore_index=True)

for name in runs:
    s = summary[summary.model == name]
    print(f"=== {name} ===")
    view = s[["category", "n", "share", "mean_entropy", "frac_multi_tier"]].copy()
    view["share"] = (view["share"] * 100).round(1).astype(str) + "%"
    view["mean_entropy"] = view["mean_entropy"].round(3)
    view["frac_multi_tier"] = (view["frac_multi_tier"] * 100).round(0).astype("Int64").astype(str) + "%"
    print(view.to_string(index=False))
    print(f"  total {int(s.n.sum())}   "
          f"later_regression (fixed then broken again): "
          f"{int(classified[name].later_regression.sum())}\n")

strict_v1:   0%|          | 0/300 [00:00<?, ?it/s]

fixed_v2:   0%|          | 0/300 [00:00<?, ?it/s]

relaxed_v3:   0%|          | 0/300 [00:00<?, ?it/s]

final_term_v4:   0%|          | 0/300 [00:00<?, ?it/s]

strict_v1:   0%|          | 0/300 [00:00<?, ?it/s]

fixed_v2:   0%|          | 0/300 [00:00<?, ?it/s]

relaxed_v3:   0%|          | 0/300 [00:00<?, ?it/s]

final_term_v4:   0%|          | 0/300 [00:00<?, ?it/s]

=== Qwen2.5-VL-3B ===
            category   n share  mean_entropy frac_multi_tier
      correct_robust 134 44.7%         0.530             39%
   bug_fix_recovered   6  2.0%         1.187             33%
   cosmetic_mismatch  22  7.3%         1.215             59%
      scope_mismatch  27  9.0%         1.310             44%
  false_pass_removed   6  2.0%         0.613             83%
broken_by_relaxation   1  0.3%         1.332            100%
     genuinely_wrong 104 34.7%         1.139             65%
  total 300   later_regression (fixed then broken again): 2

=== Pixtral-12B ===
            category   n share  mean_entropy frac_multi_tier
      correct_robust 118 39.3%         0.490             30%
   bug_fix_recovered   9  3.0%         0.858             67%
   cosmetic_mismatch  11  3.7%         1.242             45%
      scope_mismatch  25  8.3%         1.393             32%
  false_pass_removed   6  2.0%         0.908             67%
broken_by_relaxation   1  0.3%         1.33

In [5]:
# The bar plot: what the population is made of, both models side by side.
import matplotlib.pyplot as plt

FIG_DIR = f"{PROJECT_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

ax = pilot.plotting.plot_scoring_categories(summary)
ax.set_title("What the 300 scored items are actually made of\n"
             "(blue = correct under the loosest rule, orange = not)",
             fontsize=10.5, loc="left")
ax.figure.tight_layout()
ax.figure.savefig(f"{FIG_DIR}/scoring_categories.png", dpi=160)
plt.show()
print("saved -> figures/scoring_categories.png")

saved -> figures/scoring_categories.png


### Which bug caused each flip?

`bug_fix_recovered` and `false_pass_removed` are attributed by applying each
fix **alone** and seeing which one changes the verdict. Small enough numbers to
name the responsible defect per item.

In [6]:
for name, c in classified.items():
    attr = c["attributed_bug"].dropna()
    print(f"{name:15s} {attr.value_counts().to_dict()}")

print()
print("Cross-tab: attribution by category (Qwen)")
c = classified["Qwen2.5-VL-3B"]
print(pd.crosstab(c["category"], c["attributed_bug"].fillna("—")).to_string())

Qwen2.5-VL-3B   {'textcolor': 6, 'sympy_prefix': 5, 'decimal': 1}
Pixtral-12B     {'textcolor': 8, 'sympy_prefix': 6, 'decimal': 1}

Cross-tab: attribution by category (Qwen)
attributed_bug        decimal  sympy_prefix  textcolor    —
category                                                   
broken_by_relaxation        0             0          0    1
bug_fix_recovered           0             1          5    0
correct_robust              0             0          0  134
cosmetic_mismatch           0             0          0   22
false_pass_removed          1             4          1    0
genuinely_wrong             0             0          0  104
scope_mismatch              0             0          0   27


### The orthogonal flags

Extractor-tier instability is **not** a category — it cross-cuts all of them.
`extract_final_answer` has 4 tiers, and on a large share of items it fires a
*different* tier across the 5 samples, which inflates entropy without the model
having changed its mind.

**That count is a lower bound.** It only catches a changed *branch*. It misses
same-branch, different-block cases — item 9 has all five samples in
`display_math`, three returning the conclusion `(x,z) \in R` and two returning
the intermediate step, entropy 1.332 from samples that agree mathematically.

In [7]:
for name, c in classified.items():
    print(f"=== {name} ===")
    print(f"  items using >1 extractor branch: "
          f"{int((c.n_distinct_tiers > 1).sum())}/{len(c)}")
    grp = c.groupby("n_distinct_tiers").agg(
        items=("entropy", "size"),
        mean_entropy=("entropy", "mean"),
        frac_correct_strict=("correct_strict_v1", "mean")).round(3)
    print(grp.to_string())
    ends_right = c.category.isin(["correct_robust", "bug_fix_recovered",
                                  "cosmetic_mismatch", "scope_mismatch"])
    print(f"  multi-tier among items that END correct: "
          f"{(c.n_distinct_tiers > 1)[ends_right].mean():.0%}")
    print(f"  multi-tier among items that END wrong  : "
          f"{(c.n_distinct_tiers > 1)[~ends_right].mean():.0%}")
    print("  -> cross-cuts the taxonomy, so it is a flag and not a bar\n")

=== Qwen2.5-VL-3B ===
  items using >1 extractor branch: 153/300
                  items  mean_entropy  frac_correct_strict
n_distinct_tiers                                          
1                   147         0.685                0.565
2                   128         1.031                0.383
3                    25         1.243                0.360
  multi-tier among items that END correct: 42%
  multi-tier among items that END wrong  : 67%
  -> cross-cuts the taxonomy, so it is a flag and not a bar

=== Pixtral-12B ===
  items using >1 extractor branch: 94/300
                  items  mean_entropy  frac_correct_strict
n_distinct_tiers                                          
1                   206         0.822                0.417
2                    78         1.023                0.397
3                    16         1.130                0.500
  multi-tier among items that END correct: 33%
  multi-tier among items that END wrong  : 29%
  -> cross-cuts the taxonomy, so i

## 4. The viewer

`show_item` prints the handwritten image and then the full four-stage trace,
rendered by `pilot.rescore.format_trace` so the naming lives beside the scoring
and cannot drift from it. Every line says which function produced it.

The **COMPARISON** block at the bottom is the part that was missing: on a
cosmetic mismatch the two labels look identical on screen, so it reports the
exact column where they first diverge and both characters by `repr`.

In [8]:
IMAGE_DIR = f"{PROJECT_DIR}/scoring_inspection_images"
os.makedirs(IMAGE_DIR, exist_ok=True)


def show_item(i, model="Qwen2.5-VL-3B", rules=("strict_v1", "final_term_v4"),
              show_image=True, save=True, raw_chars=320):
    df = runs[model]
    row = df.iloc[i]
    cat = classified[model].loc[i]
    samples_raw = ast.literal_eval(row["all_transcription_samples_raw"])

    print("#" * 100)
    print(f"# ITEM {i}   [{model}]   category = {cat['category']}"
          + (f"   attributed to: {cat['attributed_bug']}"
             if pd.notna(cat["attributed_bug"]) else ""))
    print(f"#   has_error={bool(row['has_error'])}   "
          f"handwriting_style={row['handwriting_style']}   "
          f"image_quality={row['image_quality']}   "
          f"extractor branches used: {cat['n_distinct_tiers']}"
          + ("   later_regression=True" if cat["later_regression"] else ""))
    print("#   verdict by rule: " + "   ".join(
        f"{r}={bool(classified[model].loc[i, 'correct_' + r])}"
        for r in pilot.rescore.RULES))
    print("#" * 100)

    if show_image:
        img = sample[i]["image"]
        if save:
            img.save(f"{IMAGE_DIR}/item{i:03d}.png")
        plt.figure(figsize=(9, 9 * img.height / max(img.width, 1)))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"item {i} — the handwritten page the model was shown")
        plt.show()

    for rule in rules:
        tr = pilot.rescore.trace_item(samples_raw, row["pert_a"], rule)
        print(pilot.rescore.format_trace(
            tr, question=row["orig_q"], category=cat["category"],
            raw_chars=raw_chars))


def first_in(category, model="Qwen2.5-VL-3B", n=1):
    return classified[model].index[
        classified[model]["category"] == category].tolist()[:n]


print("show_item(i, model=...) ready.")
for name in runs:
    print(f"  {name}: " + "  ".join(
        f"{c}={len(classified[name][classified[name].category == c])}"
        for c in pilot.rescore.CATEGORIES))

show_item(i, model=...) ready.
  Qwen2.5-VL-3B: correct_robust=134  bug_fix_recovered=6  cosmetic_mismatch=22  scope_mismatch=27  false_pass_removed=6  broken_by_relaxation=1  genuinely_wrong=104
  Pixtral-12B: correct_robust=118  bug_fix_recovered=9  cosmetic_mismatch=11  scope_mismatch=25  false_pass_removed=6  broken_by_relaxation=1  genuinely_wrong=130


### 4a. `false_pass_removed` — it only *looked* correct

Read this one first. A bug mangled the ground truth and the model's samples the
same way, so they matched — and the frozen rule scored it correct. It is the
proof that relaxing a comparison is not automatically generous.

In [9]:
for i in first_in("false_pass_removed"):
    show_item(i)

####################################################################################################
# ITEM 2   [Qwen2.5-VL-3B]   category = false_pass_removed   attributed to: sympy_prefix
#   has_error=True   handwriting_style=True   image_quality=False   extractor branches used: 1
#   verdict by rule: strict_v1=True   fixed_v2=False   relaxed_v3=False   final_term_v4=False
####################################################################################################
RULE strict_v1   entropy=0.000   majority 5/5   CORRECT   [false_pass_removed]

QUESTION  (orig_q, raw — this is what the page asks)
                                  Convert \( 40^\circ 20' \) into radian measure. ⏎  ⏎

─ GROUND TRUTH ─────────────────────────────────────────────────────────────────────────────────────
     raw answer (pert_a):          We know that \( 180^\circ = \pi \) radian. ⏎  ⏎ Hence,  ⏎ \[ ⏎
                                  40^\circ 20' = 40 \frac{1}{3} \text{ degree} = \frac{\pi}{180}
   

### 4b. `cosmetic_mismatch` — the model read the page right

Watch the COMPARISON block: the two labels look identical until it names the
column where they diverge.

In [10]:
for i in first_in("cosmetic_mismatch"):
    show_item(i)

####################################################################################################
# ITEM 9   [Qwen2.5-VL-3B]   category = cosmetic_mismatch
#   has_error=False   handwriting_style=True   image_quality=True   extractor branches used: 1
#   verdict by rule: strict_v1=False   fixed_v2=False   relaxed_v3=True   final_term_v4=True
####################################################################################################
RULE strict_v1   entropy=1.055   majority 2/5   WRONG   [cosmetic_mismatch]

QUESTION  (orig_q, raw — this is what the page asks)
                                  Let \( R \) be a relation from \( \mathbb{Q} \) to \( \mathbb{Q}
                                  \) defined by \( R = \{(a,b): a,b \in \mathbb{Q} \text{ and } a -
                                  b \in \mathbb{Z}\} \). Show that ⏎ \begin{enumerate} ⏎     \item
                                  \( (a,a) \in R \) for all \( a \in \mathbb{Q} \) ⏎     \item \(
                          

### 4c. `scope_mismatch` — answer only vs the whole chain

In [11]:
for i in first_in("scope_mismatch"):
    show_item(i)

####################################################################################################
# ITEM 0   [Qwen2.5-VL-3B]   category = scope_mismatch
#   has_error=False   handwriting_style=True   image_quality=True   extractor branches used: 2
#   verdict by rule: strict_v1=False   fixed_v2=False   relaxed_v3=False   final_term_v4=True
####################################################################################################
RULE strict_v1   entropy=1.055   majority 2/5   WRONG   [scope_mismatch]

QUESTION  (orig_q, raw — this is what the page asks)
                                  Find the number of 4 letter words, with or without meaning, which
                                  can be formed out of the letters of the word ROSE, where the
                                  repetition of the letters is not allowed. ⏎  ⏎

─ GROUND TRUTH ─────────────────────────────────────────────────────────────────────────────────────
     raw answer (pert_a):          There are as m

### 4d. `bug_fix_recovered` — a real read the extractor was hiding

In [12]:
for i in first_in("bug_fix_recovered"):
    show_item(i)

####################################################################################################
# ITEM 53   [Qwen2.5-VL-3B]   category = bug_fix_recovered   attributed to: textcolor
#   has_error=True   handwriting_style=True   image_quality=True   extractor branches used: 1
#   verdict by rule: strict_v1=False   fixed_v2=True   relaxed_v3=True   final_term_v4=True
####################################################################################################
RULE strict_v1   entropy=1.332   majority 2/5   WRONG   [bug_fix_recovered]

QUESTION  (orig_q, raw — this is what the page asks)
                                  If \( nC_9 = nC_8 \), find \( nC_{17} \). ⏎  ⏎

─ GROUND TRUTH ─────────────────────────────────────────────────────────────────────────────────────
     raw answer (pert_a):          We have  ⏎ \[ ⏎ nC_9 = nC_8 ⏎ \] ⏎ i.e.,  ⏎ \[ ⏎
                                  \frac{n!}{9!(n-9)!} = \frac{n!}{(n-8)!8!} ⏎ \] ⏎ or  ⏎ \[ ⏎
                                  \

### 4e. `genuinely_wrong` — what a real misread looks like

The control for everything above. If these look like the categories above, the
taxonomy is not separating what it claims to.

In [13]:
for i in first_in("genuinely_wrong"):
    show_item(i)

####################################################################################################
# ITEM 8   [Qwen2.5-VL-3B]   category = genuinely_wrong
#   has_error=True   handwriting_style=True   image_quality=True   extractor branches used: 2
#   verdict by rule: strict_v1=False   fixed_v2=False   relaxed_v3=False   final_term_v4=False
####################################################################################################
RULE strict_v1   entropy=1.609   majority 1/5   WRONG   [genuinely_wrong]

QUESTION  (orig_q, raw — this is what the page asks)
                                  A and B together have Rs. 1210. If \(\frac{4}{15}\) of A's amount
                                  is equal to \(\frac{2}{5}\) of B's amount, how much amount does B
                                  have? ⏎  ⏎

─ GROUND TRUTH ─────────────────────────────────────────────────────────────────────────────────────
     raw answer (pert_a):          Option B ⏎  ⏎ \[ ⏎ \frac{4}{15} A = \frac{2

### 4f. The same categories on Pixtral-12B

Different model family, same failure modes — which is the point of running
both.

In [14]:
for cat in ("false_pass_removed", "cosmetic_mismatch"):
    for i in first_in(cat, model="Pixtral-12B"):
        show_item(i, model="Pixtral-12B", rules=("strict_v1",))

####################################################################################################
# ITEM 2   [Pixtral-12B]   category = false_pass_removed   attributed to: sympy_prefix
#   has_error=True   handwriting_style=True   image_quality=False   extractor branches used: 2
#   verdict by rule: strict_v1=True   fixed_v2=False   relaxed_v3=False   final_term_v4=False
####################################################################################################
RULE strict_v1   entropy=0.950   majority 3/5   CORRECT   [false_pass_removed]

QUESTION  (orig_q, raw — this is what the page asks)
                                  Convert \( 40^\circ 20' \) into radian measure. ⏎  ⏎

─ GROUND TRUTH ─────────────────────────────────────────────────────────────────────────────────────
     raw answer (pert_a):          We know that \( 180^\circ = \pi \) radian. ⏎  ⏎ Hence,  ⏎ \[ ⏎
                                  40^\circ 20' = 40 \frac{1}{3} \text{ degree} = \frac{\pi}{180}
     

## 5. Every item whose verdict a looser rule changes

For scanning after the detailed reads above. Shows the labels under the rule
that **failed** the item and the rule that **passed** it, so a "cosmetic" row
visibly has two differing labels on the left and one agreeing label on the right.

In [15]:
FAIL_PASS = {"bug_fix_recovered": ("strict_v1", "fixed_v2"),
             "cosmetic_mismatch": ("fixed_v2", "relaxed_v3"),
             "scope_mismatch": ("relaxed_v3", "final_term_v4"),
             "false_pass_removed": ("fixed_v2", "strict_v1")}

MODEL = "Qwen2.5-VL-3B"
scored_by_rule = {r: pilot.rescore.rescore_run(runs[MODEL], r)
                  for r in pilot.rescore.RULES}
pd.set_option("display.max_colwidth", 46)

for cat, (fail_rule, pass_rule) in FAIL_PASS.items():
    idx = classified[MODEL].index[classified[MODEL].category == cat]
    if not len(idx):
        continue
    rows = [{
        "i": i,
        "H": round(float(classified[MODEL].loc[i, "entropy"]), 3),
        f"{fail_rule}: model": scored_by_rule[fail_rule].loc[i, "majority_label"][:44],
        f"{fail_rule}: truth": scored_by_rule[fail_rule].loc[i, "gt_label"][:44],
        f"{pass_rule}: both": scored_by_rule[pass_rule].loc[i, "majority_label"][:44],
    } for i in idx]
    print(f"--- {cat} ({len(rows)}) — differs under {fail_rule}, "
          f"agrees under {pass_rule} " + "-" * 18)
    print(pd.DataFrame(rows).to_string(index=False))
    print()

--- bug_fix_recovered (6) — differs under strict_v1, agrees under fixed_v2 ------------------
  i     H                             strict_v1: model                             strict_v1: truth                               fixed_v2: both
 53 1.332                 text:nc_{17} = 18c_{17} = 18 text:nc_{17} = \textcolor{red}{18c_{17}} = 1                 text:nc_{17} = 18c_{17} = 18
172 1.332   text:a_n = ar^{n-1} = 5(5)^{n-1} = 5^{n-1} text:a_n = ar^{n-1} = 5(5)^{n-1} = \textcolo   text:a_n = ar^{n-1} = 5(5)^{n-1} = 5^{n-1}
206 0.950                                      sympy:p                                    sympy:1/2                                    sympy:1/2
209 0.950     text:the perimeter of the park is 1600 m                                       text:}     text:the perimeter of the park is 1600 m
232 0.950 text:\frac{1}{3^{-2}} = \frac{1}{3 \times (- text:\frac{1}{3^{-2}} = \frac{1}{3 \times (- text:\frac{1}{3^{-2}} = \frac{1}{3 \times (-
272 1.609 text:s_5 = 3(1 - (\frac{2}

## 6. What to carry out of this notebook

- **`strict_v1` remains the reported rule.** Every locked number and every
  `reference/*.json` snapshot uses it, and it is bit-identical to
  `canonical_answer_label` (locked by
  `pilot/tests/test_rescore.py::test_strict_v1_is_bit_identical_to_the_frozen_pipeline`).
- **The transcription accuracy we report is a floor, not an estimate.** Say so
  in the paper, and quote it as a **range, 47–63%**, not as 63.3% — 9 of the 30
  items `final_term_v4` newly scores correct rest on a ≤2-character match
  (`1`, `5`, `24`), where a wrong answer can land on the same token by
  coincidence. The other 21 are substantive.
- **The AUROC is robust to the scoring rule**, which is the claim the
  sensitivity table actually supports.
- **The categories replicate across model families.** Qwen and Pixtral have the
  same shape — comparable `scope_mismatch` and an identical 6-item
  `false_pass_removed` — so these are properties of the *scoring pipeline*,
  not of one model.
- **All three extractor defects are real and are now implemented correctly** —
  `canonicalize.unwrap_latex_macro`, `extract_final_answer(...,
  fix_decimal_split=True)`, and `canonicalize_math(..., strict_parse=True)`.
  The frozen entry points keep the old behaviour on purpose, each with a
  docstring saying why.
- **Watch for `sympy:` labels containing spelled-out words** (`sympy:h*(e*(n*(c*e)))`).
  That is always SymPy having parsed English as multiplied variables and
  discarded the rest of the line.
- **Relaxation is not automatically generous.** `false_pass_removed` is 6 items
  in *both* models — items a looser rule scored correct only because a bug had
  mangled both sides the same way. Fix the extractor before trusting any
  relaxed number.
- **Open, and worth a sentence in Limitations:** the extractor fires different
  branches across the five samples on a large share of items, and mean entropy
  rises with that count. Part of the perception signal may be extractor
  instability rather than model uncertainty. Distinguishing them needs a rule
  where the extractor cannot vary — e.g. requiring `\boxed{}` in the prompt —
  which is a new run, not a rescoring.

## 6. The `genuinely_wrong` audit — where the real errors are

`genuinely_wrong` is the largest category after `correct_robust` (104 Qwen /
130 Pixtral) and no scoring rule touches it. Everything above this point was
about the extractor; this section is about the model.

Offline analysis already established the shape:

- **73 items (24%) are wrong on BOTH models**, 31 Qwen-only, 57 Pixtral-only.
- On those 73 the two models **agree with each other on only 22**, so they
  misread the same pages *differently* — genuine perception failure, not a
  shared scoring artifact.
- Ground-truth length does **not** separate pages both models fail (median 439
  chars) from pages neither fails (426), so it is not "longer is harder".
- **3 Qwen / 6 Pixtral items are unanimous across all 5 samples AND wrong** —
  the cases entropy cannot help with by construction. Highest-value pages.

**Everything above is from strings. Nobody has looked at a page.** That is what
this section is for.

### Why this writes files instead of showing plots

On the 2026-08-10 run of this notebook **every `plt.show()` produced nothing**
in the synced local copy — zero `image/png` outputs, no cell error, and no
repo-side cause (no `.gitattributes`, no git filters, and no commit of this
notebook has ever contained an image). Whatever drops them, a saved PNG is
immune. The sheets below are **written to Drive** and reviewed from there.

In [16]:
# Contact sheets: one PNG per page, written to Drive. Open them from
# My Drive > uncertainty-math-vlm > figures > genuinely_wrong/
import matplotlib.pyplot as plt

AUDIT_DIR = f"{PROJECT_DIR}/figures/genuinely_wrong"
os.makedirs(AUDIT_DIR, exist_ok=True)

gw = {m: set(classified[m].index[classified[m]["category"] == "genuinely_wrong"])
      for m in runs}
both = sorted(gw["Qwen2.5-VL-3B"] & gw["Pixtral-12B"])
only = {m: sorted(gw[m] - gw[other])
        for m, other in [("Qwen2.5-VL-3B", "Pixtral-12B"),
                         ("Pixtral-12B", "Qwen2.5-VL-3B")]}

print(f"genuinely_wrong   Qwen {len(gw['Qwen2.5-VL-3B'])}   "
      f"Pixtral {len(gw['Pixtral-12B'])}")
print(f"  both models wrong : {len(both)}")
for m, idx in only.items():
    print(f"  {m} only        : {len(idx)}")

# Unanimous AND wrong: entropy 0 means all 5 samples agreed on a wrong answer,
# so this is exactly the population the uncertainty signal cannot flag.
unanimous = {m: sorted(i for i in gw[m] if classified[m].loc[i, "entropy"] < 1e-9)
             for m in runs}

# Labels for the captions. scored_by_rule above covers Qwen only, and the
# sheets need both models, so rescore the loosest rule per model here.
v4_by_model = {m: pilot.rescore.rescore_run(runs[m], "final_term_v4")
               for m in runs}

print("\nunanimous AND wrong (entropy == 0):")
for m, idx in unanimous.items():
    print(f"  {m:15s} {len(idx)}  items {idx}")

genuinely_wrong   Qwen 104   Pixtral 130
  both models wrong : 73
  Qwen2.5-VL-3B only        : 31
  Pixtral-12B only        : 57

unanimous AND wrong (entropy == 0):
  Qwen2.5-VL-3B   3  items [176, 208, 218]
  Pixtral-12B     6  items [14, 138, 180, 218, 273, 280]


In [17]:
def caption_for(i, model):
    '''Short label under each page: enough to find the item again, and to see
    whether entropy had any chance of flagging it.'''
    c = classified[model].loc[i]
    v4 = v4_by_model[model]
    tag = "BOTH models wrong" if i in both else f"{model} only"
    return (f"item {i}   H={c['entropy']:.2f}   {tag}\n"
            f"model: {str(v4.loc[i, 'majority_label'])[:38]}\n"
            f"truth: {str(v4.loc[i, 'gt_label'])[:38]}")


def write_sheets(indices, model, stem, title):
    if not indices:
        print(f"  {stem}: nothing to write")
        return
    figs = pilot.plotting.contact_sheet(
        [sample[i]["image"] for i in indices],
        [caption_for(i, model) for i in indices],
        ncols=3, per_page=12, title=title)
    for page, fig in enumerate(figs, 1):
        path = f"{AUDIT_DIR}/{stem}_p{page}.png"
        fig.savefig(path, dpi=150, facecolor=fig.get_facecolor())
        plt.close(fig)
    print(f"  {stem}: {len(indices)} items -> {len(figs)} page(s)")


print("writing contact sheets to Drive...")
# Priority 1: unanimous and wrong -- entropy cannot flag these by construction.
for m in runs:
    write_sheets(unanimous[m], m, f"unanimous_wrong_{m.split('-')[0].lower()}",
                 f"Unanimous across 5 samples AND wrong - {m}")
# Priority 2: failed by both models, so not a model-specific quirk.
write_sheets(both, "Qwen2.5-VL-3B", "wrong_on_both",
             "Wrong on BOTH models (the hard pages)")
# Priority 3: model-specific failures, for contrast.
for m, idx in only.items():
    write_sheets(idx, m, f"only_{m.split('-')[0].lower()}",
                 f"Wrong on {m} only")
print("\nopen: My Drive > uncertainty-math-vlm > figures > genuinely_wrong")

writing contact sheets to Drive...
  unanimous_wrong_qwen2.5: 3 items -> 1 page(s)
  unanimous_wrong_pixtral: 6 items -> 1 page(s)
  wrong_on_both: 73 items -> 7 page(s)
  only_qwen2.5: 31 items -> 3 page(s)
  only_pixtral: 57 items -> 5 page(s)

open: My Drive > uncertainty-math-vlm > figures > genuinely_wrong


### How to read the sheets

Work through `unanimous_wrong_*` first — nine pages across both models, and
the cases where the entropy signal is silent by construction.

For each, the question is which of these it is:

1. **Genuinely illegible handwriting** — the honest floor of the task.
2. **The model read it correctly but the ground truth is odd** — a scoring
   problem that survived all four rules, so it would need a new category.
3. **The model "corrected" the page.** Item 55 is the confirmed example:
   FERMAT's injected error is a sign flip (`1 + \tan x \tan y` where the
   identity has `1 -`), and both models transcribed the textbook-correct
   version. The prior overrode the image.

Mechanism 3 is the interesting one, and the perception-arm analogue of the
reasoning arm's documented "pattern-match the procedure instead of
recomputing" failure. **The aggregate version is NOT resolved** — the
`genuinely_wrong` rate on `has_error=1` vs clean items is +6.7%
[−4.0%, +17.3%] on both models, spanning zero — so it is a hypothesis to
pre-register on a future run, not a finding to report.